# 🏗️ Notebook 1: Amazon Lambda (Serverless) — Requirements & Architecture

Welcome! In this lab we design a **Function-as-a-Service (FaaS)** platform — the kind
of system that powers AWS Lambda, Google Cloud Functions, and Cloudflare Workers.

## 🎯 Learning Objectives

By the end of this notebook you should be able to:

1. Explain what "serverless" actually means (spoiler: there are still servers 😉).
2. List functional and non-functional requirements for a FaaS platform.
3. Do back-of-envelope math to size storage, QPS, and network.
4. Draw the **control plane vs data plane** architecture.
5. Name the four core request flows (deploy, cold start, warm start, async).

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> 💡 This lab has **no Docker / no cloud dependencies**. Everything is simulated in pure Python
> so you can see the moving parts clearly. Real Lambda uses Firecracker microVMs, S3, DynamoDB,
> and SQS under the hood — we'll point out where those fit as we go.


## 🤔 What is "serverless"?

You upload a small piece of code — a **function** — and tell the platform:
*"run this whenever an HTTP request arrives / a file is uploaded to S3 / a timer fires."*

You **don't** manage servers, VMs, containers, load balancers, or autoscalers.
You pay **per millisecond** of execution, and **nothing** when your function is idle.

That's the dream. The engineering problem: how do we run *millions* of tiny, untrusted
user programs **safely** and **cheaply**, while starting them in **under a second**?

The three forces in tension throughout this design are:

| Force | What it means |
|---|---|
| 🚀 **Fast starts** | Users' code must run quickly — cold start < 500 ms, warm start < 20 ms. |
| 🔒 **Hard isolation** | Alice's function must **never** see Bob's memory, files, or network. |
| 💰 **High density** | One physical server must run **thousands** of functions cheaply. |

Everything else in this lab is a technique for balancing these three.

## 📋 Requirements

### Functional

- **Upload code** (`.zip` + runtime: python3.11, node20, go1.21, …) with memory/timeout config.
- **Invoke synchronously** (`RequestResponse`): client waits for result.
- **Invoke asynchronously** (`Event`): fire-and-forget, retry on failure.
- **Auto-scale** from 0 to thousands of concurrent instances per function.
- **Logs** — capture `stdout`/`stderr` per invocation.
- **Versioning** — `$LATEST` + immutable numbered versions.

### Non-functional

- **Warm-start overhead**: < 20 ms.
- **Cold-start overhead**: < 500 ms for light runtimes.
- **Availability**: 99.99 % for the invocation API.
- **Isolation**: hardware-level (microVMs), not just Linux namespaces.
- **Durability**: 11 nines for stored code artifacts (S3-style).

## 🧮 Back-of-envelope estimates

Before drawing boxes, let's size the problem so later design choices are grounded in numbers.
We'll use a small runnable calculator so you can tweak the assumptions.

In [1]:
# One knob per assumption — tweak and rerun.
reqs_per_day      = 10_000_000_000   # 10 billion invocations / day
peak_factor       = 5                # traffic spikes 5× the average
active_functions  = 2_000_000        # functions that actually get called
avg_code_mb       = 30               # compressed .zip size
cold_start_ratio  = 0.01             # 1% of peak = cold start
avg_mem_mb        = 256              # memory per concurrent invocation
avg_concurrent    = 100_000          # average live (non-idle) VMs

avg_qps  = reqs_per_day / 86_400
peak_qps = avg_qps * peak_factor
storage_tb       = active_functions * avg_code_mb / 1_000_000
cold_bw_gbps     = peak_qps * cold_start_ratio * avg_code_mb / 1000
fleet_ram_tb     = avg_concurrent * avg_mem_mb / 1_000_000

print(f"Average QPS:           {avg_qps:>12,.0f}")
print(f"Peak QPS:              {peak_qps:>12,.0f}")
print(f"Code storage:          {storage_tb:>12,.1f} TB   (S3 handles this trivially)")
print(f"Cold-start bandwidth:  {cold_bw_gbps:>12,.1f} GB/s (scary — this is why we cache code on workers!)")
print(f"Fleet RAM:             {fleet_ram_tb:>12,.1f} TB   (across all physical servers)")

Average QPS:                115,741
Peak QPS:                   578,704
Code storage:                  60.0 TB   (S3 handles this trivially)
Cold-start bandwidth:         173.6 GB/s (scary — this is why we cache code on workers!)
Fleet RAM:                     25.6 TB   (across all physical servers)


**Takeaway**: the hot number is **cold-start bandwidth**. Pulling 30 MB zips from S3 for every
cold start would saturate the network. This single number justifies *code caching on worker nodes*,
*warm pools*, and *snapshots* — all covered in Notebook 3.

## 🏛️ High-level architecture

Real FaaS systems split into a **control plane** ("management") and a **data plane** ("execution").
This split is common across cloud systems (S3, EKS, DynamoDB all do it) because the two planes
have very different performance and reliability requirements.

```
┌────────────────────── CONTROL PLANE (slow & strongly consistent) ──────────────────────┐
│                                                                                         │
│   Developer ──► Function Manager ──► DynamoDB (metadata)                                │
│                        │                                                                │
│                        └──► S3 (code .zip, immutable versions)                          │
│                                                                                         │
└─────────────────────────────────────────────────────────────────────────────────────────┘

┌───────────────────── DATA PLANE (fast & eventually consistent) ────────────────────────┐
│                                                                                         │
│   Event source ──► Front-end Invoker ──► Placement Service ──► Worker Node              │
│   (HTTP, SQS,         (stateless)          (picks a slot)       ┌────────────────┐      │
│    S3, cron)                │                     │             │ MicroVM pool   │      │
│                             │                     └──► warm?─►  │  • Firecracker │      │
│                             │                               └─► │  • per-tenant  │      │
│                             ▼                                   └────────────────┘      │
│                        Async Queue (SQS/Kafka) ──► Poller loop (retry + DLQ)           │
│                                                                                         │
└─────────────────────────────────────────────────────────────────────────────────────────┘
```

| Plane | Rate | Consistency | What breaks if it's slow? |
|---|---|---|---|
| Control | ~10s QPS | Strong | Developers wait a second to deploy — annoying, not critical. |
| Data | ~600 k QPS | Eventual | **Every user's app goes down.** |

Because the data plane is the hot path, we cache function metadata on every invoker
(TTL ~60 s) so an invocation never has to hit DynamoDB.

## 🔄 The four core flows

We'll simulate each of these in Notebooks 2 and 3. Preview:

1. **Deploy** (control plane) — upload zip to S3, write metadata to DynamoDB.
2. **Cold start** (data plane) — first invocation: download code, boot microVM, init runtime, run handler (~500 ms).
3. **Warm start** (data plane) — reuse a frozen microVM that's still hot (~10 ms).
4. **Async invocation** — push event to queue, poller retries with exponential backoff, failures go to a DLQ.

## 🧱 Why these choices?

- **Stateless front-ends** (invoker, function manager): scale horizontally, die without data loss.
- **Stateful stores** (S3, DynamoDB) chosen per access pattern — big-blob vs key-value.
- **MicroVM per tenant**: each VM has its own kernel, so a kernel exploit stays inside one tenant's VM.
  Plain Linux containers (shared kernel) are too risky for multi-tenant code execution.
- **Queue for async**: smooths spikes; without it, a burst of async events would overwhelm workers.

Next up: the **data model & APIs** you actually expose to users.